In [ ]:
%matplotlib inline

import cv2
from matplotlib import pyplot as plt
import torch
import mlflow
from ultralytics import YOLO

from testing_logic_d2 import TestingLogicD2

import warnings
warnings.filterwarnings("ignore") 

torch.cuda.is_available()
torch.cuda.device_count()

In [ ]:
test = {'resize_images': False, 'resize_with_padding': False}
test['resize_images']

In [ ]:
host = "127.0.0.1"
port = "8080"
mlflow.set_tracking_uri(uri=F"http://{host}:{port}")

In [ ]:
model_type = "Yolo"
model_image_size = "1280"
model_use = "Car Detection"
hardware = "GPU"


resize_images = False
resize_with_padding = False

In [ ]:
def run_experiment(model, params, exp_tags, exp_name): 
    
    mlflow.set_experiment(exp_name)

    testing_logic_d2 = TestingLogicD2()
    metrics = testing_logic_d2.test_model(model, params=params, experiment_tags=exp_tags)
    
    num_samples = 269 if params['dataset'] == 'train' else 57
    experiment_desc = F"Dataset: Dataset_2/{params['dataset']}.\nModel Use: {model_use} \nNumber of Samples: {num_samples} "

    run_name = F"{params['model_name']}-{params['dataset']}-ccm:{params['car_choice_metric']}"
    print(run_name)
    # Start an MLflow run
    with mlflow.start_run(run_name=run_name, description=experiment_desc,tags=exp_tags):
        # Log the hyperparameters
        mlflow.log_params(params)

        # Log the metrics
        for key,val in metrics.items():
            mlflow.log_metric(key, val)


In [ ]:
exp_name = F"{model_use}. Dataset2. {model_type} Only. {hardware}. YoloV5 vs YoloV8 vs YoloV11"

for mn in ['yolo11m.pt','yolov5m','yolov8m.pt']:
    if mn == 'yolov5m':
        model = torch.hub.load('ultralytics/yolov5', mn, pretrained=True)
        model_type = 'Yolo'
    elif mn == 'yolov8m.pt' or mn == 'yolo11m.pt' :
        model = YOLO(mn) 
        model_type = 'Ultra_Yolo'

    for d in ['train','test']:
    # for d in ['train']:
        for cc_metric in ['confidence','area']:
        # for cc_metric in ['area']:
            for rim in [False]:
                for rwp in [False]:
                    car_choice_metric = cc_metric
                    dataset = d

                    params = {
                        "resize_images": rim,
                        "resize_with_padding": rwp,
                        "car_choice_metric": car_choice_metric,
                        'model_name':mn,
                        "model_type": model_type,
                        "dataset": dataset,

                    }
                    experiment_tags = {
                        "dataset": F"Dataset_2/{dataset}",
                        "model_use": model_use,
                        "hardware": F"{hardware}"
                    }
                    print(params)
                    print(experiment_tags)
                    print("\n")

                    # if rim==False and rwp==True:
                    #     continue


                    run_experiment(model=model,
                                    params=params,
                                    exp_tags=experiment_tags,
                                    exp_name=exp_name)

                

There is no need for > YoloV5

YoloV5m/YoloV5m6 will suffice
